# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boddulavinaykumar6-stack/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis**

One row represents the daily search performance of one pseudonymized content page for one pseudonymized client on one report date.

**Table**

The primary table used is `fact_content_daily_performance`.

**Time window**

This analysis uses the March 2026 partition (`month=2026-03`), covering data from **2026-03-01** to **2026-03-31**. This mid-panel month is recommended for feature development because it avoids using the final month of the dataset.

**Prediction target**

The long-term goal is to rank content pages by refresh priority. Since the warehouse does not contain a direct refresh priority label, this remains a proxy target.

**Excluded information**

Fields that contain future information or are derived from future outcomes must not be used as model features because they would introduce data leakage. IDs such as `client_hash_id` and `content_hash_id` are used only for grouping and identification, not as model features.

In [2]:
!pip -q install huggingface_hub duckdb pyarrow pandas

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
from huggingface_hub import login
import duckdb
import pandas as pd
import pyarrow.dataset as ds

# Load your Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Login to Hugging Face
login(token=HF_TOKEN)

# Create a DuckDB connection
con = duckdb.connect()

# Give DuckDB access to the Hugging Face Hub
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("✅ Hugging Face login successful.")
print("✅ DuckDB secret created.")

✅ Hugging Face login successful.
✅ DuckDB secret created.


## 2. Fields: feature / label / context / excluded

### Features
- `gsc_impressions` – Daily Google Search impressions.
- `gsc_clicks` – Daily Google Search clicks.
- `gsc_avg_position` – Daily average search position.
- `ga4_pageviews` – Daily page views when GA4 data is available.
- `ga4_sessions` – Daily sessions when GA4 data is available.

These fields are potential features because they describe page performance before a refresh decision.

### Label / Proxy
- Refresh Priority Score (proxy).

The warehouse does not contain a direct refresh priority label, so the prediction target remains a proxy.

### Context
- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

These fields identify or organize the data but should not be used as model features.

### Excluded
- `gsc_data_available`
- `ga4_data_available`

These fields indicate whether data is available. They are useful for filtering valid records but are not suitable as predictive features.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_groups = {
    "Features": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions"
    ],
    "Context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "month"
    ],
    "Excluded": [
        "gsc_data_available",
        "ga4_data_available"
    ]
}

for group, cols in feature_groups.items():
    print(f"\n{group}:")
    print(cols)


Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']

Context:
['client_hash_id', 'content_hash_id', 'report_date', 'month']

Excluded:
['gsc_data_available', 'ga4_data_available']


## 3. Verify it with queries

The following queries verify the key claims made in the data contract.

### Verification 1 — Grain

The first query checks whether any duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id` exist. An empty result confirms that each row represents one content page for one client on one report date.

### Verification 2 — Counts and time window

The second query verifies the reporting period and total number of rows in the selected March 2026 partition.

### Verification 3 — Data availability

The final query counts how many rows have Search Console and Google Analytics data available using the `IS TRUE` availability flags.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rel = "hf://datasets/FlyRank/internship-warehouse"

display(con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates


In [32]:
display(con.sql(f"""
SELECT
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day,
    COUNT(*) AS rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df())

,first_day,last_day,rows
0,2026-03-01,2026-03-31,9841378


In [33]:
display(con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df())

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 3.5 Feature frame

The following query builds a small feature frame from the March 2026 partition.

Each feature is available before a refresh decision because it summarizes search or engagement performance already observed for the content page.

In [35]:
feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
LIMIT 10
""").df()

display(feature_frame)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,1
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,2
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,2
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,1
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,1
5,2026-03-01,client_65de48885f4ef01b,content_872342e050545a12,39,0,6.538462,1,1
6,2026-03-01,client_65de48885f4ef01b,content_3c286ded8bd68120,88,1,8.431818,1,1
7,2026-03-01,client_65de48885f4ef01b,content_b2108e8fe3360fa6,40,1,5.300000,1,1
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,0,30.304348,2,1
9,2026-03-01,client_65de48885f4ef01b,content_bd07be40ea0d5f54,23,0,5.478261,1,1


### Why these features?

- **gsc_impressions** — Knowable at the decision moment because impressions are already recorded before deciding whether to refresh.
- **gsc_clicks** — Knowable because clicks are historical observations.
- **gsc_avg_position** — Knowable because search ranking is already measured.
- **ga4_pageviews** — Knowable when GA4 data is available because page views have already occurred.
- **ga4_sessions** — Knowable when GA4 data is available because sessions have already been recorded.

## 4. Data limits

One limitation of this dataset is that not every content page has complete Search Console or Google Analytics data available. The availability checks showed that only a subset of rows contains usable GSC or GA4 metrics.

As a result, analyses or models that rely on these metrics must filter the data using the availability flags (`gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`). This reduces the amount of data available for training and may limit coverage across all clients.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(availability)

print("\nObservation:")
print("- Not every row has GSC data available.")
print("- Even fewer rows have GA4 data available.")
print("- Models using these metrics must filter valid rows first.")

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966



Observation:
- Not every row has GSC data available.
- Even fewer rows have GA4 data available.
- Models using these metrics must filter valid rows first.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.